# 🔐 NeMo Safe Synthesizer Tutorial: Downstream Task Utility

#### What you'll learn

This notebook evaluates how useful synthetic data is for a downstream binary classification task. We compare two logistic regression models: one trained on real data and one trained on synthetic data.

The workflow is:
1. Load telco churn data and build a deterministic 8,000/2,000 train-holdout split.
2. Train Safe Synthesizer on the 8,000 real training rows.
3. Generate 8,000 synthetic training rows.
4. Train two identical ML pipelines (real-trained vs synthetic-trained).
5. Evaluate both models on the same 2,000-row real holdout set.

### 🖥️ Prerequisites

This notebook requires a Linux machine with an NVIDIA GPU (H100 recommended, A100 minimum) and CUDA 12.9+. It will not run on macOS, Windows, or Apple Silicon.

### ⚡ Install dependencies

Run the cell below to install NeMo Safe Synthesizer (engine + CUDA 12.9) and the Python packages used for downstream model training and evaluation.

In [ ]:
%%bash
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

if command -v uv > /dev/null 2>&1; then
    uv pip install "nemo-safe-synthesizer[engine,cu129]" --index https://flashinfer.ai/whl/cu129 --index https://download.pytorch.org/whl/cu129 --index https://wheels.vllm.ai/88d34c6409e9fb3c7b8ca0c04756f061d2099eb1/cu129 --index-strategy unsafe-best-match
    uv pip install datasets
else
    pip install "nemo-safe-synthesizer[engine,cu129]" --extra-index-url https://flashinfer.ai/whl/cu129 --extra-index-url https://download.pytorch.org/whl/cu129 --extra-index-url https://wheels.vllm.ai/88d34c6409e9fb3c7b8ca0c04756f061d2099eb1/cu129
    pip install datasets
fi

### 🔑 Set the inference API key for PII column classification

NeMo Safe Synthesizer uses an LLM-based classifier to infer potential PII columns. To enable this, set `NSS_INFERENCE_KEY` (you can create one at [build.nvidia.com](https://build.nvidia.com/settings/api-keys)).

Setting this value is optional, but strongly recommended for better PII handling quality.

In [2]:
import getpass
import os

if "NSS_INFERENCE_KEY" not in os.environ:
    os.environ["NSS_INFERENCE_KEY"] = getpass.getpass("Paste inference API key (or press Enter to skip): ")

print("NSS_INFERENCE_KEY is set" if os.environ.get("NSS_INFERENCE_KEY") else "NSS_INFERENCE_KEY is not set")

NSS_INFERENCE_KEY is set


In [3]:
os.environ["NIM_MODEL_ID"] = "qwen/qwen3-next-80b-a3b-instruct"

### 📥 Load telco dataset and create train/holdout split

Load the telco churn dataset from CSV, then create a deterministic 10,000-row subset for this tutorial.

From this subset:
- 8,000 rows are used as the real training source for Safe Synthesizer and the real-trained baseline model.
- 2,000 rows are held out as a fixed evaluation set shared by both downstream models.

The target label is `Churn Label`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("telco_churn_DD_0526.csv")
TARGET = "Churn Label"
SEED = 42

full_df = pd.read_csv(DATA_PATH)

# tmp: change NaN to None for Churn Reason
full_df["Churn Reason"] = full_df["Churn Reason"].where(full_df["Churn Reason"].notna(), None)

train_real_df, holdout_df = train_test_split(
    full_df, test_size=2000, random_state=SEED, shuffle=True
)
train_real_df = train_real_df.reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

print(f"Train rows: {len(train_real_df)}")
print(f"Holdout rows: {len(holdout_df)}")

Train rows: 8000
Holdout rows: 2000


### ⚙️ Train Safe Synthesizer and generate synthetic training data

Create a `SafeSynthesizer` builder with the 8,000-row real training split, then run the full synthesis pipeline.

In this notebook, generation is configured to produce exactly 8,000 synthetic rows so the synthetic-trained model and real-trained model use the same training set size.

In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = SafeSynthesizer().with_data_source(train_real_df).with_train(num_input_records_to_sample=10000).with_generate(num_records=8000)
builder.run()
results = builder.results
synth_df = results.synthetic_data

print(f"Synthetic rows: {len(synth_df)}")
synth_df.head()

### 🤖 Train two downstream logistic regression models

To ensure a fair utility comparison, both models use the same preprocessing and architecture:
- one model is trained on real 8,000-row training data,
- one model is trained on synthetic 8,000-row training data.

Both are evaluated on the same real 2,000-row holdout split.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DROP_COLUMNS = [
    "CustomerID",
    "Name",
    "Zip Code",
    "City",
    "State",
    "Country",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "Count",
    "Total Charges",
]

def make_xy(df: pd.DataFrame):
    y = (df[TARGET] == "Yes").astype(int)
    x = df.drop(columns=DROP_COLUMNS + [TARGET], errors="ignore")
    return x, y

def build_model(feature_df: pd.DataFrame):
    cat_cols = feature_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in feature_df.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                cat_cols,
            ),
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ]
    )

    return Pipeline(
        steps=[
            ("prep", preprocessor),
            ("clf", LogisticRegression(max_iter=5000, random_state=SEED)),
        ]
    )

x_train_real, y_train_real = make_xy(train_real_df)
x_train_synth, y_train_synth = make_xy(synth_df)
x_holdout, y_holdout = make_xy(holdout_df)

real_model = build_model(x_train_real).fit(x_train_real, y_train_real)
synth_model = build_model(x_train_synth).fit(x_train_synth, y_train_synth)

In [21]:
def evaluate_model(name, model, x_eval, y_eval):
    y_pred = model.predict(x_eval)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_eval, y_pred, average="binary", zero_division=0
    )
    return {
        "model": name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": confusion_matrix(y_eval, y_pred).tolist(),
    }

real_metrics = evaluate_model("Trained on real 8k", real_model, x_holdout, y_holdout)
synth_metrics = evaluate_model("Trained on synthetic 8k", synth_model, x_holdout, y_holdout)

comparison_df = pd.DataFrame([real_metrics, synth_metrics]).set_index("model")
display(comparison_df[["accuracy", "precision", "recall", "f1"]].round(4))

print("Confusion matrices:")
print(f"Real model:  {real_metrics['confusion_matrix']}")
print(f"Synth model: {synth_metrics['confusion_matrix']}")

,accuracy,precision,recall,f1
model,,,,
Trained on real 8k,0.8660,0.6299,0.2658,0.3738
Trained on synthetic 8k,0.8655,0.6702,0.2093,0.3190


Confusion matrices:
Real model:  [[1652, 47], [221, 80]]
Synth model: [[1668, 31], [238, 63]]


### 📊 Interpret downstream utility results

Compare the holdout metrics (accuracy, precision, recall, and F1) from both models.

If the synthetic-trained model is close to the real-trained baseline, the synthetic dataset preserves meaningful predictive signal for this task. Larger metric gaps indicate utility loss for downstream training.

In [8]:
# Synthetic data and evaluation artifacts are automatically saved to the run directory.
print(f"Artifacts saved to: {builder._workdir.generate.path}")

Artifacts saved to: safe-synthesizer-artifacts/default---data/2026-05-29T14:39:09/generate
